# Permutations II

**Problem Link:**  
https://leetcode.com/problems/permutations-ii/description/

## Problem Statement
Given a collection of numbers `nums` that might contain duplicates, return all possible **unique** permutations in any order.

## Examples

### Example 1
Input: `nums = [1,1,2]`  
Output: `[[1,1,2],[1,2,1],[2,1,1]]`

### Example 2
Input: `nums = [1,2,3]`  
Output: `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`

## Constraints
- `1 <= nums.length <= 8`
- `-10 <= nums[i] <= 10`


## Backtracking with Duplicate Pruning

### Strategy
This is Permutations I extended to handle duplicates. The key challenge is avoiding generating the same permutation multiple times when identical elements exist.

- **Sort** `nums` first — this groups duplicate values together, making it easy to detect and skip them
- Use a `used` boolean array (indexed by position, not value) to track which elements are currently in `current`
- **Pruning rule to skip duplicates:** if `nums[i] == nums[i-1]` and `used[i-1]` is `False`, skip `nums[i]`
  - This means: if the previous identical element was **not** used in the current branch, don't use the current one either
  - This enforces that among identical elements, we always pick the leftmost available one first — guaranteeing each unique permutation is generated exactly once
- **Base case:** when `current` is full, record the permutation

### Why track by index (not value)?
Since there can be duplicate values, a value-based `used` set would incorrectly block placing a second copy of the same value. Using index-based tracking lets us distinguish between two elements that happen to share the same value.

### Time Complexity
- **O(n × n!)** in the worst case (all distinct), with significant pruning when duplicates exist

### Space Complexity
- **O(n)** for the recursion stack, `current`, and `used` array


In [1]:
from typing import List

class Solution:
    def permuteUnique(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        result = []
        used   = [False] * len(nums)

        def backtrack(current: list):
            if len(current) == len(nums):
                result.append(list(current))
                return
            for i in range(len(nums)):
                if used[i]:
                    continue
                # Skip duplicate: same value as previous AND previous was not used in this branch
                if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                    continue
                used[i] = True
                current.append(nums[i])
                backtrack(current)
                current.pop()
                used[i] = False

        backtrack([])
        return result


In [2]:
from math import factorial
from collections import Counter

def count_unique_permutations(nums):
    """n! / (c1! * c2! * ... ck!) where ci are the counts of each distinct value."""
    n   = len(nums)
    res = factorial(n)
    for cnt in Counter(nums).values():
        res //= factorial(cnt)
    return res

def test_permute_unique():
    sol = Solution()

    # Example 1
    result = sol.permuteUnique([1, 1, 2])
    assert sorted(map(tuple, result)) == sorted(map(tuple, [[1,1,2],[1,2,1],[2,1,1]]))

    # Example 2: no duplicates — same as Permutations I
    result = sol.permuteUnique([1, 2, 3])
    assert len(result) == 6
    assert len(set(map(tuple, result))) == 6

    # All same elements
    result = sol.permuteUnique([2, 2, 2])
    assert result == [[2, 2, 2]]

    # Single element
    assert sol.permuteUnique([5]) == [[5]]

    # Count check using multinomial formula
    for nums in [[1,1,2], [1,2,3], [1,1,1,2], [1,2,2,3]]:
        result   = sol.permuteUnique(nums)
        expected = count_unique_permutations(nums)
        assert len(result) == expected, f"Expected {expected} unique perms for {nums}, got {len(result)}"

    # No duplicate permutations in output
    result = sol.permuteUnique([1, 1, 2])
    assert len(result) == len(set(map(tuple, result))), "Duplicate permutations found"

    # Each permutation contains exactly the original elements
    for perm in sol.permuteUnique([1, 1, 2]):
        assert sorted(perm) == [1, 1, 2], f"Permutation {perm} has wrong elements"

    print("All test cases passed!")

test_permute_unique()


All test cases passed!
